# Bronze → Silver · V-Credit

## Objetivo
Aplicar as regras de negócio de cada base e gravar tabelas tipadas, padronizadas e confiáveis na camada Silver.

## Ordem de execução
1. Bases de referência: canais, motivos, atendentes e clientes
2. Chamados_Hora, chamados, custos e pesquisa de satisfação, que dependem das bases de referência

## Log de qualidade
Cada regra registra quantas linhas afetou. O log é gravado em `tb_log_qualidade` ao final do notebook e é a base da leitura de confiabilidade dos dados.

In [0]:
from pyspark.sql import functions as F

catalogo = "vcredit"
bronze = f"{catalogo}.bronze"
silver = f"{catalogo}.silver"

# Datas fora do padrão viram NULL em vez de gerar erro de versão do parser
spark.conf.set("spark.sql.legacy.timeParserPolicy", "CORRECTED")

# Dicionário por (base, regra): reexecutar uma célula atualiza o registro em vez de duplicar
log_qualidade = {}


def registrar(base, regra, linhas_afetadas, total_linhas):
    log_qualidade[(base, regra)] = (int(linhas_afetadas), int(total_linhas))


def chave_texto(coluna):
    # Minúsculas, sem acento e sem espaços extras: base para comparar textos digitados de formas diferentes
    com_acento = "áàâãäéèêëíìîïóòôõöúùûüç"
    sem_acento = "aaaaaeeeeiiiiooooouuuuc"
    return F.trim(F.regexp_replace(F.translate(F.lower(coluna), com_acento, sem_acento), r"\s+", " "))


def de_para(dicionario):
    # Converte um dicionário Python em um mapa do Spark para traduzir valores coluna a coluna
    return F.create_map([F.lit(valor) for par in dicionario.items() for valor in par])


def gravar_silver(df, tabela):
    # Overwrite garante a idempotência: reexecutar substitui a tabela em vez de duplicar
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver}.{tabela}")
    print(f"{silver}.{tabela}: {spark.table(f'{silver}.{tabela}').count()} linhas")

## Funções complementares

- **vazio:** identifica valores NULL ou textos vazios, inclusive quando contêm apenas espaços
- **carregar:** lê a tabela Bronze, confere os metadados e remove somente linhas de negócio idênticas
- **registrar_condicoes:** conta as linhas afetadas por cada regra e registra os resultados usando a função de log original
- **salvar_log:** grava o diagnóstico de qualidade antes de interromper o processamento por uma inconsistência
- **exigir:** verifica uma condição de erro; quando encontra registros inválidos, salva o log e interrompe o processamento
- **validar_unicidade:** verifica chaves ausentes ou repetidas; conflitos interrompem o processamento, sem escolher um registro ao acaso
- **Referências:** canais, motivos, atendentes e clientes são conferidos antes dos novos joins, evitando multiplicação de registros
- **Fuso horário:** definido como America/Sao_Paulo para interpretar os horários dos chamados

In [0]:
# Nomes das colunas de rastreabilidade criadas na Bronze.
# ingestion_datetime informa quando o registro foi carregado; arquivo_origem identifica o CSV.
# A lista é uma variável Python, não uma configuração obrigatória do Databricks.
# Nos selects, *metadados inclui as duas colunas sem repetir seus nomes.
metadados = ["ingestion_datetime", "arquivo_origem"]

# Interpreta os horários dos chamados no fuso usado pelo case.
spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")

# Ausência: identifica NULL ou texto vazio depois de remover espaços nas pontas.
def vazio(coluna):
    return coluna.isNull() | (F.trim(coluna) == "")

# Log: conta várias regras na mesma consulta e reaproveita a função registrar original.
def registrar_condicoes(df, base, regras):
    # Cada condição vale 1 quando verdadeira e 0 nos outros casos. A soma conta as linhas afetadas.
    # first() traz somente o resumo calculado, sem trazer a base inteira para o Python.
    resumo = df.agg(F.count("*").alias("total"), *[
        F.sum(F.when(F.coalesce(condicao, F.lit(False)), 1).otherwise(0)).alias(f"regra_{i}")
        for i, (_, condicao) in enumerate(regras)
    ]).first()
    # Grava cada contagem no dicionário de qualidade; uma soma vazia é registrada como zero.
    for i, (regra, _) in enumerate(regras):
        registrar(base, regra, resumo[f"regra_{i}"] or 0, resumo["total"])


# Diagnóstico: grava as regras já avaliadas antes de interromper por um erro de conteúdo.
def salvar_log():
    # Mesmo contrato de quatro campos do dicionario original.
    # Transforma o dicionário no mesmo formato de tabela usado pelo log original.
    df_log = spark.createDataFrame(
        [(base, regra, afetadas, total) for (base, regra), (afetadas, total) in log_qualidade.items()],
        "base string, regra string, linhas_afetadas int, total_linhas int",
    ).withColumn("percentual", F.when(F.col("total_linhas") > 0,
        F.round(100.0 * F.col("linhas_afetadas") / F.col("total_linhas"), 2)))
    df_log = df_log.withColumn("data_execucao", F.current_timestamp())
    gravar_silver(df_log, "tb_log_qualidade")

# Bloqueio: registra as linhas que descumprem uma regra e impede continuar com o problema.
def exigir(df, base, condicao_invalida, regra, total_base=None):
    # total_base permite usar o total da base completa quando df contém apenas um subconjunto.
    total = df.count() if total_base is None else total_base
    afetadas = df.filter(F.coalesce(condicao_invalida, F.lit(False))).count()
    registrar(base, regra, afetadas, total)
    if afetadas:
        # Salva o diagnóstico antes de interromper. Não escolhe registros ao acaso.
        salvar_log()
        raise ValueError(f"{base}: {regra}; {afetadas} linhas. Consulte tb_log_qualidade.")

# Chaves: identifica valores NULL e registros que repetem a mesma chave.
def validar_unicidade(df, base, chaves):
    falta = None
    # Combina as verificações: basta uma das colunas da chave estar NULL para haver problema.
    for chave in chaves:
        condicao = F.col(chave).isNull()
        falta = condicao if falta is None else falta | condicao
    exigir(df, base, falta, "Chave obrigatoria ausente ou invalida")
    # Agrupa pela chave e procura grupos com mais de uma linha. Não escolhe uma versão ao acaso.
    conflitos = df.groupBy(*chaves).count().filter(F.col("count") > 1)
    # Conta as linhas dos grupos repetidos, e não apenas a quantidade de chaves repetidas.
    afetadas = conflitos.agg(F.sum("count")).first()[0] or 0
    registrar(base, "Linhas em chaves repetidas com conflito", afetadas, df.count())
    if afetadas:
        salvar_log()
        raise ValueError(f"{base}: chave repetida com conteudos diferentes; falta uma regra de escolha.")

# Leitura dos blocos novos: confere a tabela, mantém a origem e remove repetições idênticas.
def carregar(base):
    if not spark.catalog.tableExists(f"{bronze}.{base}"):
        raise ValueError(f"Falta {bronze}.{base}. Execute a Bronze completa.")
    df = spark.table(f"{bronze}.{base}")
    if not all(coluna in df.columns for coluna in metadados):
        raise ValueError(f"{base}: faltam metadados da Bronze recebida.")
    # Deduplica somente linhas de negócio idênticas. A Bronze é um snapshot completo,
    # com metadados iguais dentro de cada carga; metadados não definem a duplicidade.
    # Data de carga e arquivo de origem não definem se o conteúdo de negócio está repetido.
    negocio = [c for c in df.columns if c not in metadados]
    total = df.count()
    resultado = df.dropDuplicates(negocio)
    registrar(base, "Linhas de negocio identicas removidas", total - resultado.count(), total)
    return resultado


## 1. tb_canais

- **Nome:** padronizado para URA, Chatbot, Atendimento Inicial e Atendimento Especializado
- **Tipo:** padronizado para Autosserviço e Humano
- **Nível:** nova coluna com a etapa do funil (Autosserviço, N1 ou N2)
- **Duplicatas:** removidas por `id_canal`, como proteção para cargas futuras
- Um canal fora do de-para interrompe o pipeline, pois indica um canal novo na origem

In [0]:
# De-para do nome normalizado (só letras) para o nome padronizado
nomes_canais = {
    "ura": "URA",
    "chatbot": "Chatbot",
    "atendimentoinicial": "Atendimento Inicial",
    "atendimentoespecializado": "Atendimento Especializado",
}
tipos_canais = {"autosservico": "Autosserviço", "humano": "Humano"}

# Etapa do funil de cada canal
niveis_canais = {
    "URA": "Autosserviço",
    "Chatbot": "Autosserviço",
    "Atendimento Inicial": "N1",
    "Atendimento Especializado": "N2",
}

df_bronze = spark.table(f"{bronze}.tb_canais")
total = df_bronze.count()

# Duplicatas: remoção por id_canal como proteção para cargas futuras
df_bronze = df_bronze.dropDuplicates(["id_canal"])
registrar("tb_canais", "Duplicatas por id_canal removidas", total - df_bronze.count(), total)

df_canais = (
    df_bronze
    .select(
        # Tipagem: id como inteiro
        F.col("id_canal").cast("int").alias("id_canal"),
        # Nome: mantém só as letras do texto normalizado e aplica o de-para
        de_para(nomes_canais)[F.regexp_replace(chave_texto(F.col("nome_canal")), "[^a-z]", "")].alias("nome_canal"),
        # Tipo: "Autosservico" ganha o acento e fica Autosserviço
        de_para(tipos_canais)[chave_texto(F.col("tipo_canal"))].alias("tipo_canal"),
        # Status: em minúsculas
        F.lower(F.trim(F.col("status_canal"))).alias("status_canal"),
    )
    # Nível: nova coluna com a etapa do funil
    .withColumn("nivel_canal", de_para(niveis_canais)[F.col("nome_canal")])
)

# Canal fora do de-para indica um canal novo na origem: o pipeline para em vez de seguir com NULL
if df_canais.filter(F.col("nome_canal").isNull() | F.col("tipo_canal").isNull()).count() > 0:
    raise ValueError("tb_canais possui canal ou tipo fora do de-para.")

gravar_silver(df_canais, "tb_canais")

## 2. tb_motivos

- **Categoria:** grafia padronizada (Financeiro, Cartão, Cadastral, Atendimento, Benefícios). Os 3 motivos sem categoria são classificados pelos critérios do case: Contratação de cartão adicional em Cartão, Transferência de agência em Cadastral e Cancelamento de conta em Financeiro
- **Criticidade:** padronizada para Baixa, Média e Alta
- **chave_motivo:** nome do motivo em minúsculas, sem acento e sem espaços extras, usado para ligar o texto livre dos chamados ao `id_motivo`
- **Duplicatas:** removidas por `id_motivo`

In [0]:
# De-para da categoria normalizada para a grafia padronizada
categorias = {
    "financeiro": "Financeiro",
    "cartao": "Cartão",
    "cadastral": "Cadastral",
    "atendimento": "Atendimento",
    "beneficios": "Benefícios",
}

# Motivos sem categoria na origem, classificados pelos critérios do case
categorias_sem_origem = {
    "contratacao de cartao adicional": "Cartão",
    "transferencia de agencia": "Cadastral",
    "cancelamento de conta": "Financeiro",
}
criticidades = {"baixa": "Baixa", "media": "Média", "alta": "Alta"}

df_bronze = spark.table(f"{bronze}.tb_motivos")
total = df_bronze.count()

# Duplicatas: remoção por id_motivo como proteção para cargas futuras
df_bronze = df_bronze.dropDuplicates(["id_motivo"])
registrar("tb_motivos", "Duplicatas por id_motivo removidas", total - df_bronze.count(), total)

# Chave de associação: nome normalizado, usado para ligar o texto livre dos chamados ao id_motivo
df_motivos = df_bronze.withColumn("chave_motivo", chave_texto(F.col("nome_motivo")))

# Registro no log das inconsistências encontradas antes do tratamento
sem_categoria = F.col("categoria").isNull() | (F.trim(F.col("categoria")) == "")
registrar("tb_motivos", "Categoria vazia classificada pelos critérios do case", df_motivos.filter(sem_categoria).count(), total)
registrar(
    "tb_motivos",
    "Categoria com grafia fora do padrão",
    df_motivos.filter(~sem_categoria & ~F.col("categoria").isin(list(categorias.values()))).count(),
    total,
)
registrar(
    "tb_motivos",
    "Criticidade com grafia fora do padrão",
    df_motivos.filter(~F.col("criticidade").isin(list(criticidades.values()))).count(),
    total,
)

df_motivos = df_motivos.select(
    # Tipagem: id como inteiro
    F.col("id_motivo").cast("int").alias("id_motivo"),
    F.trim(F.col("nome_motivo")).alias("nome_motivo"),
    # Categoria: vazia é classificada pelo motivo; preenchida tem a grafia padronizada
    F.when(sem_categoria, de_para(categorias_sem_origem)[F.col("chave_motivo")])
    .otherwise(de_para(categorias)[chave_texto(F.col("categoria"))])
    .alias("categoria"),
    # Criticidade: padronizada para Baixa, Média e Alta
    de_para(criticidades)[chave_texto(F.col("criticidade"))].alias("criticidade"),
    "chave_motivo",
)

# Categoria ou criticidade fora do de-para interrompe o pipeline
if df_motivos.filter(F.col("categoria").isNull() | F.col("criticidade").isNull()).count() > 0:
    raise ValueError("tb_motivos possui categoria ou criticidade fora do de-para.")

gravar_silver(df_motivos, "tb_motivos")

## 3. tb_atendentes

- **Nível:** padronizado para 1 e 2 a partir das variações (1, Nível 1, N2, Nível 2)
- **Datas:** admissão e desligamento convertidas de dd/MM/aaaa; desligamento vazio vira NULL
- **Status:** recalculado pela data de desligamento (com data, inativo; sem data, ativo), com registro no log se divergir da origem
- **Duplicatas:** removidas por `id_atendente`

In [0]:
df_bronze = spark.table(f"{bronze}.tb_atendentes")
total = df_bronze.count()

# Duplicatas: remoção por id_atendente como proteção para cargas futuras
df_bronze = df_bronze.dropDuplicates(["id_atendente"])
registrar("tb_atendentes", "Duplicatas por id_atendente removidas", total - df_bronze.count(), total)

# Registro no log dos níveis escritos fora do padrão numérico
registrar(
    "tb_atendentes",
    "Nível de atendimento fora do padrão (Nível 1, N2...)",
    df_bronze.filter(~F.trim(F.col("nivel_atendimento")).rlike(r"^[12]$")).count(),
    total,
)

df_atendentes = (
    df_bronze
    .select(
        # Tipagem: id como inteiro
        F.col("id_atendente").cast("int").alias("id_atendente"),
        F.trim(F.col("nome_atendente")).alias("nome_atendente"),
        # Nível: extrai o dígito de qualquer variação (1, Nível 1, N2...)
        F.regexp_extract(F.col("nivel_atendimento"), r"([12])", 1).cast("int").alias("nivel_atendimento"),
        # Datas: dd/MM/aaaa para date; desligamento vazio vira NULL
        F.to_date(F.try_to_timestamp(F.trim(F.col("data_admissao")), F.lit("dd/MM/yyyy"))).alias("data_admissao"),
        F.to_date(F.try_to_timestamp(F.trim(F.col("data_desligamento")), F.lit("dd/MM/yyyy"))).alias("data_desligamento"),
        F.lower(F.trim(F.col("status_atendente"))).alias("status_origem"),
    )
    # Status: recalculado pela data de desligamento
    .withColumn("status_atendente", F.when(F.col("data_desligamento").isNotNull(), "inativo").otherwise("ativo"))
)

# Registro no log de status da origem que não batem com a data de desligamento
registrar(
    "tb_atendentes",
    "Status divergente da data de desligamento",
    df_atendentes.filter(F.col("status_origem") != F.col("status_atendente")).count(),
    total,
)

df_atendentes = df_atendentes.drop("status_origem")

# Nível ou data de admissão inválidos interrompem o pipeline
if df_atendentes.filter(F.col("nivel_atendimento").isNull() | F.col("data_admissao").isNull()).count() > 0:
    raise ValueError("tb_atendentes possui nível ou data de admissão inválidos.")

gravar_silver(df_atendentes, "tb_atendentes")

## 4. tb_clientes

- **Região:** espaços removidos e associada por de-para às 5 regiões (Norte, Nordeste, Centro-Oeste, Sudeste e Sul)
- **Data de cadastro:** convertida a partir dos formatos dd/MM/aaaa e aaaa-MM-dd
- **Idade:** válida entre 18 e 100 anos; fora disso vira NULL (a base tem idades 0 e acima de 120)
- **Faixa etária:** nova coluna (18 a 29, 30 a 44, 45 a 59, 60 a 74, 75 ou mais, Não informado)
- **E-mail:** sem "@" vira NULL
- **Dia de vencimento:** convertido para inteiro
- **id_cliente:** mantido como texto de 11 dígitos, preservando zeros à esquerda

In [0]:
# De-para da região normalizada; evita o initcap, que geraria "Centro-oeste"
regioes = {
    "norte": "Norte",
    "nordeste": "Nordeste",
    "centro-oeste": "Centro-Oeste",
    "sudeste": "Sudeste",
    "sul": "Sul",
}

df_bronze = spark.table(f"{bronze}.tb_clientes")
total = df_bronze.count()

# Duplicatas: remoção por id_cliente como proteção para cargas futuras
df_bronze = df_bronze.dropDuplicates(["id_cliente"])
registrar("tb_clientes", "Duplicatas por id_cliente removidas", total - df_bronze.count(), total)

# Idade convertida só quando o texto é numérico, para não gerar erro de conversão
idade_numero = F.when(F.trim(F.col("idade")).rlike(r"^\d+$"), F.trim(F.col("idade")).cast("int"))
data_iso = F.trim(F.col("data_cadastro")).rlike(r"^\d{4}-\d{2}-\d{2}$")

# Registro no log das inconsistências encontradas antes do tratamento
registrar(
    "tb_clientes",
    "Região com grafia fora do padrão (maiúsculas ou espaços)",
    df_bronze.filter(~F.col("regiao").isin(list(regioes.values()))).count(),
    total,
)
registrar("tb_clientes", "Data de cadastro no formato aaaa-MM-dd", df_bronze.filter(data_iso).count(), total)
registrar("tb_clientes", "Idade ausente", df_bronze.filter(idade_numero.isNull()).count(), total)
registrar("tb_clientes", "Idade fora de 18 a 100 convertida para NULL", df_bronze.filter(~idade_numero.between(18, 100)).count(), total)
registrar("tb_clientes", "E-mail sem @ convertido para NULL", df_bronze.filter(~F.col("email").contains("@")).count(), total)

# Idade: válida entre 18 e 100; fora disso vira NULL
idade = F.when(idade_numero.between(18, 100), idade_numero)

df_clientes = (
    df_bronze
    .select(
        # Chave: id_cliente como texto, preservando zeros à esquerda
        F.trim(F.col("id_cliente")).alias("id_cliente"),
        F.trim(F.col("nome")).alias("nome"),
        # E-mail: sem "@" é inválido e vira NULL
        F.when(F.col("email").contains("@"), F.trim(F.col("email"))).alias("email"),
        # Região: espaços removidos e de-para para as 5 regiões
        de_para(regioes)[chave_texto(F.col("regiao"))].alias("regiao"),
        idade.alias("idade"),
        # Faixa etária: nova coluna; idade NULL fica como Não informado
        F.when(idade < 30, "18 a 29")
        .when(idade < 45, "30 a 44")
        .when(idade < 60, "45 a 59")
        .when(idade < 75, "60 a 74")
        .when(idade.isNotNull(), "75 ou mais")
        .otherwise("Não informado")
        .alias("faixa_etaria"),
        # Data de cadastro: tenta dd/MM/aaaa e, se não casar, aaaa-MM-dd
        F.coalesce(
            F.to_date(F.try_to_timestamp(F.trim(F.col("data_cadastro")), F.lit("dd/MM/yyyy"))),
            F.to_date(F.try_to_timestamp(F.trim(F.col("data_cadastro")), F.lit("yyyy-MM-dd"))),
        ).alias("data_cadastro"),
        # Dia de vencimento: convertido para inteiro
        F.col("dia_vencimento_fatura").cast("int").alias("dia_vencimento_fatura"),
    )
)

# id sem 11 dígitos, região fora do de-para ou data inválida interrompem o pipeline
if df_clientes.filter(
    ~F.col("id_cliente").rlike(r"^\d{11}$") | F.col("regiao").isNull() | F.col("data_cadastro").isNull()
).count() > 0:
    raise ValueError("tb_clientes possui id, região ou data de cadastro inválidos.")

gravar_silver(df_clientes, "tb_clientes")

## 5. tb_chamados_hora

- **Datas:** convertidas do formato “dd/MM/aaaa às HH:mm:ss” para timestamp
- **Ano da abertura:** quando maior que o ano da finalização, recebe o ano da finalização, desde que a nova data seja válida e não fique depois do início ou fim
- **Virada de meia-noite:** finalização anterior ao início, na mesma data, recebe um dia a mais
- **Desistência:** nova coluna; verdadeiro quando o horário de início está vazio na origem
- **Horário inválido:** texto preenchido que não pode ser convertido é sinalizado, sem ser considerado desistência
- **Tempos:** espera e atendimento recalculados em segundos; ficam NULL nas desistências ou quando os horários são inválidos ou inconsistentes
- **Horários originais:** preservados em colunas de texto para conferir as correções
- **id_cliente:** mantido como texto de 11 dígitos e conferido na base de clientes
- **id_chamado:** convertido para inteiro e validado como chave única

In [0]:
# Referencias unicas sao necessarias para nao multiplicar os joins dos novos blocos.
validar_unicidade(df_canais, "tb_canais", ["id_canal"])
validar_unicidade(df_canais, "tb_canais", ["nome_canal"])
validar_unicidade(df_motivos, "tb_motivos", ["id_motivo"])
validar_unicidade(df_motivos, "tb_motivos", ["chave_motivo"])
validar_unicidade(df_atendentes, "tb_atendentes", ["id_atendente"])
validar_unicidade(df_clientes, "tb_clientes", ["id_cliente"])

# 1. Ler os horários da Bronze e remover somente repetições idênticas.
df = carregar("tb_chamados_hora")
# 2. Tipar o chamado, preservar o cliente como texto e guardar os horários originais.
# *metadados mantém ingestion_datetime e arquivo_origem na tabela tratada.
df_horas = df.select(
    F.expr("try_cast(trim(id_chamado) AS INT)").alias("id_chamado"),
    F.trim("id_cliente").alias("id_cliente"),
    F.col("hora_abertura_chamado").alias("hora_abertura_original"),
    F.col("hora_inicio_atendimento").alias("hora_inicio_original"),
    F.col("hora_finalizacao_atendimento").alias("hora_finalizacao_original"), *metadados,
)
# 3. Converter os textos para timestamp. Conteúdo inválido vira NULL com try_to_timestamp.
for origem, destino in [
    ("hora_abertura_original", "hora_abertura_chamado"),
    ("hora_inicio_original", "hora_inicio_atendimento"),
    ("hora_finalizacao_original", "hora_finalizacao_atendimento"),
]:
    df_horas = df_horas.withColumn(destino, F.try_to_timestamp(F.trim(F.col(origem)), F.lit("dd/MM/yyyy 'às' HH:mm:ss")))

# 4. Aplicar as correções de datas e criar as flags antes de calcular as durações.
df_horas = (
    # Início vazio indica desistência; texto preenchido inválido será tratado como erro de horário.
    df_horas.withColumn("desistencia", vazio(F.col("hora_inicio_original")))
    .withColumn("_ano_posterior", F.coalesce(F.year("hora_abertura_chamado") > F.year("hora_finalizacao_atendimento"), F.lit(False)))
    # Monta uma candidata com o ano da finalização e o mês, dia e horário da abertura original.
    .withColumn("_abertura_candidata", F.try_to_timestamp(
        F.concat(F.year("hora_finalizacao_atendimento").cast("string"), F.date_format("hora_abertura_chamado", "-MM-dd HH:mm:ss")),
        F.lit("yyyy-MM-dd HH:mm:ss")))
    # Só aceita uma data existente, com ano corrigido, que não fique depois do início ou fim.
    .withColumn("flag_ano_abertura_corrigido", F.coalesce(
        F.col("_ano_posterior") & F.col("_abertura_candidata").isNotNull()
        & (F.col("_abertura_candidata") <= F.coalesce(F.col("hora_inicio_atendimento"), F.col("hora_finalizacao_atendimento"))), F.lit(False)))
    .withColumn("hora_abertura_chamado", F.when(F.col("flag_ano_abertura_corrigido"), F.col("_abertura_candidata")).otherwise(F.col("hora_abertura_chamado")))
    # Finalização que volta no relógio, na mesma data do início, pode representar virada de meia-noite.
    .withColumn("flag_virada_meia_noite", F.coalesce(
        (F.col("hora_finalizacao_atendimento") < F.col("hora_inicio_atendimento"))
        & (F.to_date("hora_finalizacao_atendimento") == F.to_date("hora_inicio_atendimento")), F.lit(False)))
    .withColumn("hora_finalizacao_atendimento", F.when(F.col("flag_virada_meia_noite"),
        F.expr("hora_finalizacao_atendimento + INTERVAL 1 DAY")).otherwise(F.col("hora_finalizacao_atendimento")))
    # Diferencia desistência de um horário obrigatório ausente ou que não pôde ser interpretado.
    .withColumn("flag_horario_invalido", F.col("hora_abertura_chamado").isNull()
        | F.col("hora_finalizacao_atendimento").isNull()
        | (~F.col("desistencia") & F.col("hora_inicio_atendimento").isNull()))
    # Após os ajustes, verifica se a ordem abertura, início e finalização continua inconsistente.
    .withColumn("flag_cronologia_invalida", F.coalesce(
        (F.col("hora_inicio_atendimento") < F.col("hora_abertura_chamado"))
        | (F.col("hora_finalizacao_atendimento") < F.coalesce(F.col("hora_inicio_atendimento"), F.col("hora_abertura_chamado"))), F.lit(False)))
    # Calcula tempos somente para atendimentos iniciados com horários válidos e ordenados.
    .withColumn("tempos_validos", ~F.col("desistencia") & ~F.col("flag_horario_invalido") & ~F.col("flag_cronologia_invalida"))
    # Timestamp convertido para long representa segundos; a subtração calcula a duração em segundos.
    .withColumn("tempo_espera", F.when(F.col("tempos_validos"),
        F.col("hora_inicio_atendimento").cast("long") - F.col("hora_abertura_chamado").cast("long")))
    .withColumn("tempo_atendimento", F.when(F.col("tempos_validos"),
        F.col("hora_finalizacao_atendimento").cast("long") - F.col("hora_inicio_atendimento").cast("long")))
)
# 5. Contar as correções e inconsistências no log usando as flags criadas acima.
registrar_condicoes(df_horas, "tb_chamados_hora", [
    ("Ano da abertura corrigido com apoio da finalizacao", F.col("flag_ano_abertura_corrigido")),
    ("Ano inconsistente sem correcao segura", F.col("_ano_posterior") & ~F.col("flag_ano_abertura_corrigido")),
    ("Finalizacao ajustada na virada da meia-noite", F.col("flag_virada_meia_noite")),
    ("Desistencia por inicio vazio na origem", F.col("desistencia")),
    ("Horario obrigatorio ausente ou invalido", F.col("flag_horario_invalido")),
    ("Cronologia inconsistente mantida sem calcular tempos", F.col("flag_cronologia_invalida")),
    ("Tempos recalculados a partir dos horarios", F.col("tempos_validos")),
])
# 6. Remover somente auxiliares do cálculo e conferir as chaves finais.
df_horas = df_horas.drop("_ano_posterior", "_abertura_candidata")
validar_unicidade(df_horas, "tb_chamados_hora", ["id_chamado"])
exigir(df_horas, "tb_chamados_hora", (F.col("id_chamado") <= 0) | F.col("id_cliente").isNull()
       | ~F.col("id_cliente").rlike(r"^[0-9]{11}$"), "Chave do chamado ou do cliente invalida")
# left_anti retorna apenas os horários cujo cliente não foi encontrado no cadastro.
clientes_ausentes = df_horas.join(df_clientes.select("id_cliente"), "id_cliente", "left_anti")
exigir(clientes_ausentes, "tb_chamados_hora", F.lit(True), "Cliente dos horarios sem cadastro de referencia", df_horas.count())
# 7. Gravar a base de horários tratada na Silver.
gravar_silver(df_horas, "tb_chamados_hora")


## 6. tb_chamados

- **Duplicatas:** linhas idênticas removidas; id_chamado passa a ser único
- **Integração com horários:** id_cliente, horários, tempos e desistência vêm da tb_chamados_hora; os horários e tempos da base de chamados não são utilizados no cálculo
- **Cliente original:** preservado em id_cliente_original; diferenças em relação à base de horários são sinalizadas
- **Motivo:** texto preservado em motivo_original, normalizado e associado ao id_motivo da base de referência
- **Canal:** texto preservado em canal_original; variações de escrita, espaços, pontuação e abreviações são associadas ao id_canal
- **Resolvido:** Sim, S e 1 viram Sim; Não, N, 0 e Nao resolvido viram Não; vazio ou conteúdo desconhecido vira Não informado
- **Atendente:** IDs inexistentes na referência ficam NULL; em URA e Chatbot também ficam NULL, por serem autosserviço
- **Vínculo do atendente:** nos canais humanos, sinaliza atribuição antes da admissão ou depois do desligamento, considerando a abertura corrigida do chamado; o registro é mantido
- **Nível do atendente:** incompatibilidade com o nível do canal é sinalizada, sem alterar o registro
- **Mês de referência:** nova coluna com o primeiro dia do mês da abertura
- **Rastreabilidade:** mantém os metadados dos chamados e acrescenta os metadados da base de horários

In [0]:

# 1. Reaproveitar o de-para original e acrescentar as abreviações encontradas nos chamados.
# **nomes_canais copia os itens do dicionário, sem alterar o dicionário original.
nomes_canais_chamados = {
    **nomes_canais,
    "atendinicial": "Atendimento Inicial",
    "atendespecializado": "Atendimento Especializado",
}

# 2. Definir equivalências dos valores de resolução para Sim ou Não.
resolucoes = {"sim": "Sim", "s": "Sim", "1": "Sim", "nao": "Não", "n": "Não", "0": "Não", "nao resolvido": "Não"}
# 3. Ler os chamados, remover repetições idênticas e guardar o total antes dos joins.
df = carregar("tb_chamados")
total_chamados = df.count()
# Preserva os textos e IDs originais para comparar com os valores tratados.
# Os horários e tempos dos chamados não são usados: virão da base oficial de horários.
df_chamados = df.select(
    F.expr("try_cast(trim(id_chamado) AS INT)").alias("id_chamado"),
    F.trim("id_cliente").alias("id_cliente_original"),
    F.col("motivo").alias("motivo_original"), F.col("canal").alias("canal_original"),
    F.col("resolvido").alias("resolvido_original"), F.col("id_atendente").alias("id_atendente_original"),
    F.expr("try_cast(trim(id_atendente) AS INT)").alias("_id_atendente_lido"), *metadados,
)
validar_unicidade(df_chamados, "tb_chamados", ["id_chamado"])

# 4. Selecionar os horários e renomear seus metadados para distinguir as duas origens.
# _hora_localizada permite identificar se o join encontrou o registro correspondente.
horas_integracao = df_horas.select(
    "id_chamado", "id_cliente", "hora_abertura_chamado", "hora_inicio_atendimento", "hora_finalizacao_atendimento",
    "tempo_espera", "tempo_atendimento", "desistencia", "tempos_validos", "flag_horario_invalido",
    "flag_cronologia_invalida", "flag_ano_abertura_corrigido", "flag_virada_meia_noite",
    F.col("arquivo_origem").alias("arquivo_origem_horarios"),
    F.col("ingestion_datetime").alias("ingestion_datetime_horarios"), F.lit(True).alias("_hora_localizada"),
)
# left preserva os chamados; a validação seguinte bloqueia os que não tiverem horários correspondentes.
df_chamados = df_chamados.join(horas_integracao, "id_chamado", "left")
exigir(df_chamados, "tb_chamados", F.col("_hora_localizada").isNull(), "Chamado sem registro na base oficial de horarios")

# 5. Associar os textos livres às referências e criar as sinalizações de qualidade.
df_chamados = (
    # Normaliza os textos para procurar o motivo e o canal cadastrados.
    df_chamados.withColumn("chave_motivo", chave_texto(F.col("motivo_original")))
    .withColumn("_nome_canal", de_para(nomes_canais_chamados)[F.regexp_replace(chave_texto(F.col("canal_original")), "[^a-z0-9]", "")])
    .withColumn("resolvido", F.coalesce(de_para(resolucoes)[chave_texto(F.col("resolvido_original"))], F.lit("Não informado")))
    # As referências fornecem os IDs oficiais; left mantém o chamado para verificar a integridade.
    .join(df_motivos.select("chave_motivo", "id_motivo", F.col("nome_motivo").alias("motivo")), "chave_motivo", "left")
    .join(df_canais.select(F.col("nome_canal").alias("_nome_canal"), "id_canal", "tipo_canal", "nivel_canal"), "_nome_canal", "left")
    .withColumn("canal", F.col("_nome_canal"))
    # Procura o atendente informado e traz nível e datas do vínculo para as verificações.
    .join(df_atendentes.select(
        F.col("id_atendente").alias("_id_atendente_lido"), F.col("nivel_atendimento").alias("_nivel_atendente"),
        "data_admissao", "data_desligamento", F.lit(True).alias("_atendente_localizado")), "_id_atendente_lido", "left")
    # Compara o cliente original com o cliente recebido da base oficial de horários.
    .withColumn("flag_cliente_corrigido", ~F.col("id_cliente_original").eqNullSafe(F.col("id_cliente")))
    # ID inexistente é sinalizado nos canais humanos; em autosserviço o atendente não se aplica.
    .withColumn("flag_atendente_inexistente", F.coalesce((F.col("tipo_canal") == "Humano") & F.col("_atendente_localizado").isNull(), F.lit(False)))
    .withColumn("flag_atendente_removido_autosservico", F.coalesce((F.col("tipo_canal") == "Autosserviço") & ~vazio(F.col("id_atendente_original")), F.lit(False)))
    # Mantém o ID somente quando o canal é humano e o atendente existe na referência.
    .withColumn("id_atendente", F.when((F.col("tipo_canal") == "Humano") & F.col("_atendente_localizado").isNotNull(), F.col("_id_atendente_lido")))
    # Compara a abertura corrigida com admissão/desligamento. Os dias de admissão e desligamento são aceitos.
    # Sinaliza sem excluir o chamado; sem atendente válido para avaliar, a flag fica NULL.
    .withColumn("flag_vinculo_atendente_invalido", F.when(F.col("id_atendente").isNotNull() & F.col("hora_abertura_chamado").isNotNull(),
        F.coalesce((F.to_date("hora_abertura_chamado") < F.col("data_admissao"))
            | (F.to_date("hora_abertura_chamado") > F.col("data_desligamento")), F.lit(False))))
    # N1 espera atendente de nível 1 e N2 espera nível 2; divergências são apenas sinalizadas.
    .withColumn("flag_nivel_atendente_incompativel", F.when(F.col("id_atendente").isNotNull(),
        ((F.col("nivel_canal") == "N1") & (F.col("_nivel_atendente") != 1))
        | ((F.col("nivel_canal") == "N2") & (F.col("_nivel_atendente") != 2))))
    # Guarda possíveis contradições entre resolução e desistência, sem sobrescrever a resposta.
    .withColumn("flag_resolvido_em_desistencia", F.col("desistencia") & (F.col("resolvido") == "Sim"))
    # O primeiro dia do mês facilita agrupamentos e o relacionamento com o calendário da Gold.
    .withColumn("mes_referencia", F.trunc(F.to_date("hora_abertura_chamado"), "month"))
)
exigir(df_chamados, "tb_chamados", (F.col("id_chamado") <= 0) | F.col("id_canal").isNull()
       | F.col("id_motivo").isNull(), "ID invalido ou motivo/canal sem associacao de referencia")
# 6. Registrar as quantidades afetadas pelas padronizações e flags.
registrar_condicoes(df_chamados, "tb_chamados", [
    ("Cliente substituido pela base oficial de horarios", F.col("flag_cliente_corrigido")),
    ("Canal padronizado pelo de-para", ~F.col("canal_original").eqNullSafe(F.col("canal"))),
    ("Grafia do motivo padronizada pela referencia", ~F.col("motivo_original").eqNullSafe(F.col("motivo"))),
    ("Resolucao informada fora do dominio convertida para Nao informado", ~vazio(F.col("resolvido_original")) & (F.col("resolvido") == "Não informado")),
    ("Resolucao ausente na origem", vazio(F.col("resolvido_original"))),
    ("Grafia ou codigo da resolucao padronizado", ~vazio(F.col("resolvido_original")) & ~F.col("resolvido_original").eqNullSafe(F.col("resolvido"))),
    ("Atendente removido de autosservico", F.col("flag_atendente_removido_autosservico")),
    ("Atendente humano inexistente convertido para NULL", F.col("flag_atendente_inexistente")),
    ("Atendimento fora do vinculo do atendente mantido", F.col("flag_vinculo_atendente_invalido")),
    ("Nivel do atendente incompativel com o canal", F.col("flag_nivel_atendente_incompativel")),
    ("Chamado resolvido com desistencia mantido para analise", F.col("flag_resolvido_em_desistencia")),
])
# 7. Selecionar a estrutura final com flags, valores originais e as duas origens de dados.
df_chamados = df_chamados.select(
    "id_chamado", "id_cliente", "id_motivo", "id_canal", "id_atendente", "motivo", "canal", "tipo_canal", "nivel_canal", "resolvido",
    "hora_abertura_chamado", "hora_inicio_atendimento", "hora_finalizacao_atendimento", "tempo_espera", "tempo_atendimento",
    "desistencia", "tempos_validos", "mes_referencia", "motivo_original", "canal_original", "resolvido_original",
    "id_cliente_original", "id_atendente_original", "flag_cliente_corrigido", "flag_atendente_inexistente",
    "flag_atendente_removido_autosservico", "flag_vinculo_atendente_invalido", "flag_nivel_atendente_incompativel",
    "flag_resolvido_em_desistencia", "flag_horario_invalido", "flag_cronologia_invalida", "flag_ano_abertura_corrigido", "flag_virada_meia_noite",
    *metadados, "arquivo_origem_horarios", "ingestion_datetime_horarios",
)
validar_unicidade(df_chamados, "tb_chamados", ["id_chamado"])
# Os joins devem manter uma linha por chamado; mudança de quantidade precisa ser investigada.
if df_chamados.count() != total_chamados:
    raise ValueError("O join alterou o numero de chamados. Gold bloqueada.")
# 8. Gravar os chamados tratados na Silver.
gravar_silver(df_chamados, "tb_chamados")


## 7. tb_custos

- **Duplicatas:** linhas idênticas removidas; deve existir no máximo um custo por chamado
- **Conflitos:** custos diferentes para o mesmo chamado interrompem o processamento
- **Valor:** remoção de “R$”, “reais” e espaços; vírgula decimal convertida para ponto; convertido para decimal com 2 casas
- **Negativos:** convertidos para valor absoluto pela regra de erro de sinal definida para este case; o valor original e a flag da correção são mantidos
- **Zero:** considerado custo não registrado; vira NULL e é registrado no log
- **Valor inválido:** conteúdo que não pode ser convertido fica NULL e é sinalizado
- **Integridade:** só entram custos de chamados existentes na tb_chamados
- **Chaves:** id_custo e id_chamado convertidos para inteiro e validados

In [0]:
# 1. Ler os custos, tipar as chaves e guardar o texto original para rastrear as correções.
df = carregar("tb_custos").select(
    F.expr("try_cast(trim(id_custo) AS INT)").alias("id_custo"),
    F.expr("try_cast(trim(id_chamado) AS INT)").alias("id_chamado"),
    F.col("custo").alias("custo_original"), *metadados,
)
# 2. Conferir as chaves antes do tratamento monetário.
validar_unicidade(df, "tb_custos", ["id_custo"])
exigir(df, "tb_custos", (F.col("id_custo") <= 0) | F.col("id_chamado").isNull()
       | (F.col("id_chamado") <= 0), "Chave do custo ou do chamado invalida")
total_custos = df.count()
# 3. left_semi mantém somente custos de chamados existentes, sem trazer colunas do chamado.
df_custos = df.join(df_chamados.select("id_chamado"), "id_chamado", "left_semi")
registrar("tb_custos", "Custo de chamado inexistente removido", total_custos - df_custos.count(), total_custos)
# 4. Limpar o texto e converter para decimal, evitando cálculos monetários com float.
df_custos = (
    # Remove moeda, palavra reais e espaços; o valor original permanece em outra coluna.
    df_custos.withColumn("_custo_texto", F.regexp_replace(F.lower(F.trim("custo_original")), r"r\$|reais|\s", ""))
    # Se existe vírgula decimal, remove pontos de milhar antes de trocar a vírgula por ponto.
    .withColumn("_custo_texto", F.when(F.col("_custo_texto").contains(","),
        F.regexp_replace(F.regexp_replace("_custo_texto", r"\.", ""), ",", ".")).otherwise(F.col("_custo_texto")))
    # Confere o formato numérico e usa try_cast para conteúdo inválido virar NULL.
    .withColumn("_custo_numero", F.when(F.col("_custo_texto").rlike(r"^[+-]?[0-9]+(\.[0-9]+)?$"),
        F.expr("try_cast(_custo_texto AS DECIMAL(18,2))")))
    # Registra os casos de erro de sinal, zero não registrado e formato inválido.
    .withColumn("flag_custo_negativo_corrigido", F.coalesce(F.col("_custo_numero") < 0, F.lit(False)))
    .withColumn("flag_custo_zero_nao_registrado", F.coalesce(F.col("_custo_numero") == 0, F.lit(False)))
    .withColumn("flag_custo_invalido", F.col("_custo_numero").isNull())
    # Pela premissa do case, negativos recebem valor absoluto; zero vira NULL, não custo gratuito.
    .withColumn("custo", F.when(F.col("_custo_numero") != 0, F.abs("_custo_numero")).cast("decimal(18,2)"))
)
# 5. Registrar quantas linhas receberam cada tratamento.
registrar_condicoes(df_custos, "tb_custos", [
    ("Custo negativo corrigido pela premissa de erro de sinal", F.col("flag_custo_negativo_corrigido")),
    ("Custo zero nao registrado convertido para NULL", F.col("flag_custo_zero_nao_registrado")),
    ("Custo ausente ou formato invalido mantido como NULL", F.col("flag_custo_invalido")),
])
# 6. Manter as colunas finais e conferir no máximo um custo por chamado.
df_custos = df_custos.select("id_custo", "id_chamado", "custo", "custo_original", "flag_custo_negativo_corrigido", "flag_custo_zero_nao_registrado", "flag_custo_invalido", *metadados)
validar_unicidade(df_custos, "tb_custos", ["id_chamado"])
# 7. Gravar os custos tratados na Silver.
gravar_silver(df_custos, "tb_custos")


## 8. tb_pesquisa_satisfacao

- **Nota:** inteiros de 1 a 5; valores fora dessa faixa ou inválidos viram NULL
- **Nota ausente:** NULL textual e vazio são tratados como ausência
- **Pesquisa duplicada:** registros do mesmo chamado com a mesma nota são consolidados, mantendo o maior id_pesquisa
- **Notas conflitantes:** pesquisas do mesmo chamado com notas diferentes interrompem o processamento
- **Pesquisa em desistência:** mantida e sinalizada no log
- **Satisfeito:** nova coluna; verdadeiro para notas 4 e 5, falso para 1, 2 e 3 e NULL quando não há nota válida
- **Integridade:** só entram pesquisas de chamados existentes na tb_chamados
- **Chaves:** id_pesquisa e id_chamado convertidos para inteiro e validados

In [0]:
# 1. Ler a pesquisa, tipar as chaves e preservar a nota recebida na origem.
df = carregar("tb_pesquisa_satisfacao").select(
    F.expr("try_cast(trim(id_pesquisa) AS INT)").alias("id_pesquisa"),
    F.expr("try_cast(trim(id_chamado) AS INT)").alias("id_chamado"),
    F.col("nota_atendimento").alias("nota_original"), *metadados,
)
# 2. Conferir as chaves antes de associar as pesquisas aos chamados.
validar_unicidade(df, "tb_pesquisa_satisfacao", ["id_pesquisa"])
exigir(df, "tb_pesquisa_satisfacao", (F.col("id_pesquisa") <= 0) | F.col("id_chamado").isNull()
       | (F.col("id_chamado") <= 0), "Chave da pesquisa ou do chamado invalida")
total_pesquisas = df.count()
# 3. Manter somente pesquisas de chamados existentes e contar as exclusões no log.
df_pesquisa = df.join(df_chamados.select("id_chamado"), "id_chamado", "left_semi")
registrar("tb_pesquisa_satisfacao", "Pesquisa de chamado inexistente removida", total_pesquisas - df_pesquisa.count(), total_pesquisas)
# 4. Converter a nota e diferenciar ausência, valor inválido e nota válida.
df_pesquisa = (
    df_pesquisa.withColumn("_nota_numero", F.expr("try_cast(trim(nota_original) AS INT)"))
    .withColumn("flag_nota_ausente", vazio(F.col("nota_original")) | (chave_texto(F.col("nota_original")) == "null"))
    .withColumn("nota_atendimento", F.when(F.col("_nota_numero").between(1, 5), F.col("_nota_numero")))
    .withColumn("flag_nota_invalida", ~F.col("flag_nota_ausente") & F.col("nota_atendimento").isNull())
    # Permite comparar respostas sem confundir ausência com uma nota inválida informada.
    .withColumn("_chave_nota", F.when(F.col("flag_nota_ausente"), "SEM_NOTA")
        .when(F.col("nota_atendimento").isNotNull(), F.col("nota_atendimento").cast("string"))
        .otherwise(F.concat(F.lit("INVALIDA:"), F.trim("nota_original"))))
)
# 5. Notas diferentes para o mesmo chamado bloqueiam a carga, sem escolher resposta ao acaso.
conflitos = df_pesquisa.groupBy("id_chamado").agg(F.countDistinct("_chave_nota").alias("notas_distintas")).filter(F.col("notas_distintas") > 1)
# Recupera as linhas conflitantes para contar os registros afetados, e não apenas os grupos.
linhas_conflitantes = df_pesquisa.join(conflitos.select("id_chamado"), "id_chamado", "inner")
exigir(linhas_conflitantes, "tb_pesquisa_satisfacao", F.lit(True), "Pesquisas do mesmo chamado com notas conflitantes", df_pesquisa.count())

total_antes_consolidacao = df_pesquisa.count()
# 6. Como as notas repetidas são iguais, usa o maior ID somente como representante.
# ID maior não está sendo interpretado como resposta mais recente.
pesquisas_escolhidas = df_pesquisa.groupBy("id_chamado").agg(F.max("id_pesquisa").alias("id_pesquisa"))
df_pesquisa = (
    df_pesquisa.join(pesquisas_escolhidas, ["id_chamado", "id_pesquisa"], "inner")
    # Traz a desistência para sinalizar essas pesquisas, sem descartá-las.
    .join(df_chamados.select("id_chamado", "desistencia"), "id_chamado", "left")
    .withColumn("flag_pesquisa_em_desistencia", F.col("desistencia"))
    # TRUE para 4/5, FALSE para 1/2/3 e NULL sem nota válida; ausência não vira insatisfação no CSAT.
    .withColumn("satisfeito", F.when(F.col("nota_atendimento").isNotNull(), F.col("nota_atendimento") >= 4))
)
registrar("tb_pesquisa_satisfacao", "Pesquisas redundantes com mesma nota consolidadas pelo maior ID", total_antes_consolidacao - df_pesquisa.count(), total_antes_consolidacao)
# 7. Contar ausências, notas inválidas e pesquisas em desistência após a consolidação.
registrar_condicoes(df_pesquisa, "tb_pesquisa_satisfacao", [
    ("Nota ausente mantida como NULL", F.col("flag_nota_ausente")),
    ("Nota fora de 1 a 5 ou formato invalido convertida para NULL", F.col("flag_nota_invalida")),
    ("Pesquisa em desistencia mantida e sinalizada", F.col("flag_pesquisa_em_desistencia")),
])
# 8. Selecionar a estrutura final e conferir uma pesquisa por chamado.
df_pesquisa = df_pesquisa.select("id_pesquisa", "id_chamado", "nota_atendimento", "satisfeito", "nota_original", "flag_nota_ausente", "flag_nota_invalida", "flag_pesquisa_em_desistencia", *metadados)
validar_unicidade(df_pesquisa, "tb_pesquisa_satisfacao", ["id_chamado"])
# 9. Gravar a pesquisa tratada na Silver.
gravar_silver(df_pesquisa, "tb_pesquisa_satisfacao")


## Contrato para a Gold

`tb_chamados` tem uma linha por chamado; custos e pesquisas têm no máximo uma
linha por chamado, mas podem não cobrir todos eles. Use LEFT JOIN para manter
os chamados sem custo ou nota. Não transforme esses NULLs em zero.
TME/TMA usam somente `tempos_validos`; CSAT usa somente notas válidas.

## Log de qualidade

Consolida as regras registradas ao longo do notebook em `tb_log_qualidade`.

In [0]:
# Consolida as regras registradas no notebook; deve ser sempre a última célula
df_log = (
    spark.createDataFrame(
        [(base, regra, afetadas, total) for (base, regra), (afetadas, total) in log_qualidade.items()],
        "base string, regra string, linhas_afetadas int, total_linhas int",
    )
    # Percentual da base afetado por cada regra
    .withColumn("percentual", F.round(F.col("linhas_afetadas") / F.col("total_linhas") * 100, 2))
    .withColumn("data_execucao", F.current_timestamp())
)

gravar_silver(df_log, "tb_log_qualidade")
display(df_log)